# DSA4153 Week 10: Four-Format Data Recovery

This notebook imports the project modules rather than reimplementing the processing logic. The input data are historical Gapminder observations for 1952–2007. Run from the repository root after installing `requirements.txt`.

**Reconciliation target:** per input, 1,712 raw records = 6 rejected + 2 exact duplicate copies removed + 1,704 canonical rows. Across four formats, 24 quarantined records and 8 duplicate copies are expected.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from src.recovery import recover_one, assert_agreement
from src.analysis import create_analyses

ROOT = Path.cwd()
manifest = json.loads((ROOT / "data/raw/manifest.json").read_text(encoding="utf-8"))
config = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))
print("Canonical columns:", manifest["canonical_order"])
print("Allowed years:", manifest["allowed_years"])
print("Tolerance:", config["comparison"])

In [ ]:
bundles = [
    recover_one(ROOT / "data/raw" / filename, manifest)
    for filename in config["inputs"]
]
canonical = assert_agreement(bundles, atol=config["comparison"]["absolute_tolerance"])
pd.DataFrame([
    {"source": b.source_file, "raw": b.raw_count, "quarantined": len(b.quarantine),
     "duplicate_copies_removed": len(b.duplicates), "canonical_rows": len(b.canonical)}
    for b in bundles
])

## Agreement checks

The recovery function sorts each version by `(country, year)` and the agreement function compares keys, text, integer values, and floating-point values with absolute tolerance `1e-8` and relative tolerance `0`. A conflicting key is an error; the program must not keep the first version silently.

In [ ]:
analyses = create_analyses(canonical, int(config.get("reporting_year", 2007)))
print("Canonical shape:", canonical.shape)
print("Asian selected-year rows:", len(analyses["asia_selected_year"]))
print("Philippine history rows:", len(analyses["country_history"]))
display(analyses["asia_selected_year"].head())
display(analyses["country_history"])
display(analyses["population_index"])

## NumPy population index

The matrix uses Philippines, Indonesia, and Thailand for 1982, 1987, 1992, 1997, 2002, and 2007. Each row is divided by its 1982 baseline and multiplied by 100. The processing module demonstrates broadcasting, a Boolean selection and aggregation, and a separately written loop calculation to check the result.

In [ ]:
years = [1982, 1987, 1992, 1997, 2002, 2007]
print("Population index table shape (including country label):", analyses["population_index"].shape)
print("Broadcast matrix shape:", analyses["population_index"].shape[0], "x", len(years))
print("2007 Asia population:", int(canonical.loc[(canonical.continent == "Asia") & (canonical.year == 2007), "pop"].sum()))
print("Philippines 2007 population:", int(canonical.loc[(canonical.country == "Philippines") & (canonical.year == 2007), "pop"].iloc[0]))

## Interpretation and exports

- The four source files are alternative representations of the same observations, not four independent datasets to concatenate.
- Quarantined records are invalid classroom additions; they are preserved with their source filename, one-based data-record position, reason, and original field values.
- Exact duplicates are logged, while conflicting `(country, year)` keys stop processing.
- The command-line program writes the canonical exports and analysis reports to a staging folder, reloads all four canonical formats, and promotes the results only when the validation report passes.
- Change `reporting_year` in `config.json` to `1997` and rerun to regenerate the Asian selected-year analysis without editing processing functions.

In [ ]:
# Re-run the complete staged program from a terminal:
# python main.py --config config.json
# Export this notebook to HTML after running all cells:
# jupyter nbconvert --to html notebooks/report.ipynb --output report.html